# ColPali / ColQwen2 — Data Preparation




In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = Path('/content/drive/MyDrive/ColPali IR project')

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT_ROOT}"
    )

print("Drive mounted.")
print("Project:", PROJECT_ROOT)


Mounted at /content/drive
Drive mounted.
Project: /content/drive/MyDrive/ColPali IR project


In [ ]:
# Show the project contents so you can confirm Colab sees your files.

for p in sorted(PROJECT_ROOT.iterdir(), key=lambda x: x.name.lower()):
    print("[DIR] " if p.is_dir() else "[FILE]", p)


[DIR]  /content/drive/MyDrive/ColPali IR project/ColPali Paper
[DIR]  /content/drive/MyDrive/ColPali IR project/Data
[DIR]  /content/drive/MyDrive/ColPali IR project/Notebooks
[DIR]  /content/drive/MyDrive/ColPali IR project/Plan


## Find Markdown pages and the question CSV

The search is recursive.

The main page corpus is expected to be `.md` files.


In [ ]:
def recursive_files(root, suffixes):
    suffixes = {s.lower() for s in suffixes}
    result = []
    for p in root.rglob('*'):
        try:
            if p.is_file() and p.suffix.lower() in suffixes:
                result.append(p)
        except (PermissionError, OSError):
            pass
    return sorted(result, key=lambda x: str(x).lower())

md_files = recursive_files(PROJECT_ROOT, {'.md'})
csv_files = recursive_files(PROJECT_ROOT, {'.csv'})
image_files = recursive_files(PROJECT_ROOT, {'.png', '.jpg', '.jpeg', '.webp'})

print(f"Markdown files: {len(md_files)}")
for p in md_files[:100]:
    print("MD :", p)

print(f"\nCSV files: {len(csv_files)}")
for p in csv_files[:100]:
    print("CSV:", p)

print(f"\nImage files: {len(image_files)}")
for p in image_files[:100]:
    print("IMG:", p)


Markdown files: 310
MD : /content/drive/MyDrive/ColPali IR project/Data/Pages/Texts/paper_01/paper_01_page_001.md
MD : /content/drive/MyDrive/ColPali IR project/Data/Pages/Texts/paper_01/paper_01_page_002.md
MD : /content/drive/MyDrive/ColPali IR project/Data/Pages/Texts/paper_01/paper_01_page_003.md
MD : /content/drive/MyDrive/ColPali IR project/Data/Pages/Texts/paper_01/paper_01_page_004.md
MD : /content/drive/MyDrive/ColPali IR project/Data/Pages/Texts/paper_01/paper_01_page_005.md
MD : /content/drive/MyDrive/ColPali IR project/Data/Pages/Texts/paper_01/paper_01_page_006.md
MD : /content/drive/MyDrive/ColPali IR project/Data/Pages/Texts/paper_01/paper_01_page_007.md
MD : /content/drive/MyDrive/ColPali IR project/Data/Pages/Texts/paper_01/paper_01_page_008.md
MD : /content/drive/MyDrive/ColPali IR project/Data/Pages/Texts/paper_01/paper_01_page_009.md
MD : /content/drive/MyDrive/ColPali IR project/Data/Pages/Texts/paper_01/paper_01_page_010.md
MD : /content/drive/MyDrive/ColPali IR p

## Select the Markdown folder and question CSV




In [ ]:
from pathlib import Path

# Programmatically select the base Markdown texts folder and the queries CSV
TEXT_DIR = Path('/content/drive/MyDrive/ColPali IR project/Data/Pages/Texts')
QUERY_CSV = Path('/content/drive/MyDrive/ColPali IR project/Data/Queries/colpali_50_queries.csv')

print("Selected text folder:", TEXT_DIR)
print("Selected question CSV:", QUERY_CSV)

Selected text folder: /content/drive/MyDrive/ColPali IR project/Data/Pages/Texts
Selected question CSV: /content/drive/MyDrive/ColPali IR project/Data/Queries/colpali_50_queries.csv


## Build `page_text_df`

Every `.md` file becomes one row:

```text
page_id    page_text
```

`page_id` is exactly the Markdown filename without `.md`.


In [ ]:
import pandas as pd

# Read all markdown files from all paper folders recursively under the base text directory
page_files = sorted(list(TEXT_DIR.rglob('*.md')), key=lambda p: p.name.lower())

rows = []
for md_path in page_files:
    with open(md_path, 'r', encoding='utf-8', errors='replace') as f:
        page_text = f.read()

    rows.append({
        'page_id': md_path.stem,
        'page_text': page_text
    })

page_text_df = pd.DataFrame(rows, columns=['page_id', 'page_text'])

print(f"Loaded {len(page_text_df)} Markdown pages.")
display(page_text_df.head())

if not page_text_df['page_id'].is_unique:
    print("Warning: Duplicate page IDs detected, resolving duplicates...")
    page_text_df = page_text_df.drop_duplicates(subset=['page_id'])

print("Empty page texts:", (page_text_df['page_text'].str.strip() == '').sum())

Loaded 310 Markdown pages.


,page_id,page_text
0,paper_01_page_001,SimCSE: Simple Contrastive Learning of Sentenc...
1,paper_01_page_002,Our supervised SimCSE builds upon the recent s...
2,paper_01_page_003,Positive instances. One critical question in c...
3,paper_01_page_004,Dropout noise as data augmentation. We view it...
4,paper_01_page_005,We train the contrastive learning model (Eq. 1...


Empty page texts: 0


In [ ]:
print("Page-text length statistics:")
display(page_text_df['page_text'].str.len().describe())

print("\nExample page IDs:")
for pid in page_text_df['page_id'].head(20):
    print(pid)


Page-text length statistics:


,page_text
count,310.000000
mean,3218.612903
std,1546.581108
min,8.000000
25%,2232.000000
50%,3465.500000
75%,4173.500000
max,13053.000000



Example page IDs:
paper_01_page_001
paper_01_page_002
paper_01_page_003
paper_01_page_004
paper_01_page_005
paper_01_page_006
paper_01_page_007
paper_01_page_008
paper_01_page_009
paper_01_page_010
paper_01_page_011
paper_01_page_012
paper_01_page_013
paper_01_page_014
paper_01_page_015
paper_01_page_016
paper_01_page_017
paper_02_page_001
paper_02_page_002
paper_02_page_003


## Load the 50-question CSV

Expected columns:

`questions`, `paper`, `page_no`, `question_type`

They will become:

`qid`, `question`, `question_type`, `page_id`


In [ ]:
queries_raw = pd.read_csv(QUERY_CSV)

print("Shape:", queries_raw.shape)
print("Columns:", list(queries_raw.columns))
display(queries_raw.head())

required = {'questions', 'paper', 'page_no', 'question_type'}
missing = required - set(queries_raw.columns)
if missing:
    raise ValueError(f"Missing expected columns: {sorted(missing)}")

queries = queries_raw.copy()
queries['qid'] = [f'q{i:03d}' for i in range(1, len(queries) + 1)]
queries['question'] = queries['questions'].astype(str).str.strip()
queries['paper'] = queries['paper'].astype(str).str.strip()
queries['page_no'] = queries['page_no'].astype(int)
queries['question_type'] = queries['question_type'].astype(str).str.strip().str.lower()

print("\nQuestion types:")
display(queries['question_type'].value_counts())


Shape: (50, 4)
Columns: ['questions', 'paper', 'page_no', 'question_type']


,questions,paper,page_no,question_type
0,What role does dropout play in the unsupervise...,paper_01.pdf,2,text
1,Which dropout probability gives the strongest ...,paper_01.pdf,4,table
2,Why are entailment pairs from SNLI and MNLI us...,paper_01.pdf,5,text
3,How does first-last average pooling compare wi...,paper_01.pdf,8,table
4,Why does SPLADE retain advantages of tradition...,paper_02.pdf,2,text



Question types:


,count
question_type,
text,18
table,17
figure,15


## Match `paper + page_no` to the real Markdown filename

The notebook tries common filename patterns. It will **not silently guess** if a question cannot be matched.


In [ ]:
import re

actual_page_ids = set(page_text_df['page_id'])

def normalize(s):
    return re.sub(r'[^a-z0-9]+', '', str(s).lower())

normalized = {}
for pid in actual_page_ids:
    normalized.setdefault(normalize(pid), []).append(pid)

def candidates(paper, page_no):
    stem = re.sub(r'\.pdf$', '', str(paper), flags=re.I)
    page = int(page_no)

    out = [
        f'{stem}_page_{page:03d}',
        f'{stem}_page_{page:02d}',
        f'{stem}_page_{page}',
        f'{stem}_{page:03d}',
        f'{stem}_{page:02d}',
        f'{stem}_{page}',
    ]

    m = re.search(r'(\d+)', stem)
    if m:
        n = int(m.group(1))
        for pv in [f'paper_{n:02d}', f'paper_{n}', f'paper{n:02d}', f'paper{n}']:
            out += [
                f'{pv}_page_{page:03d}',
                f'{pv}_page_{page:02d}',
                f'{pv}_page_{page}',
                f'{pv}_{page:03d}',
                f'{pv}_{page:02d}',
                f'{pv}_{page}',
            ]

    return list(dict.fromkeys(out))

def resolve_page_id(paper, page_no):
    for c in candidates(paper, page_no):
        if c in actual_page_ids:
            return c

    for c in candidates(paper, page_no):
        matches = normalized.get(normalize(c), [])
        if len(matches) == 1:
            return matches[0]

    return None

queries['page_id'] = [
    resolve_page_id(p, n)
    for p, n in zip(queries['paper'], queries['page_no'])
]

unmatched = queries[queries['page_id'].isna()]

print(f"Matched {len(queries) - len(unmatched)} / {len(queries)} questions.")

if len(unmatched):
    print("\nUNMATCHED QUESTIONS:")
    display(unmatched[['qid', 'questions', 'paper', 'page_no', 'question_type']])

    print("\nFirst actual page IDs:")
    for pid in sorted(actual_page_ids)[:100]:
        print(pid)

    raise ValueError(
        "Some questions could not be matched to Markdown page IDs."
    )


Matched 50 / 50 questions.


In [ ]:
queries_clean = queries[
    ['qid', 'question', 'question_type', 'page_id']
].copy()

print("Final questions:", queries_clean.shape)
display(queries_clean.head(10))

print("\nQuestions by type:")
display(queries_clean['question_type'].value_counts())


Final questions: (50, 4)


,qid,question,question_type,page_id
0,q001,What role does dropout play in the unsupervise...,text,paper_01_page_002
1,q002,Which dropout probability gives the strongest ...,table,paper_01_page_004
2,q003,Why are entailment pairs from SNLI and MNLI us...,text,paper_01_page_005
3,q004,How does first-last average pooling compare wi...,table,paper_01_page_008
4,q005,Why does SPLADE retain advantages of tradition...,text,paper_02_page_002
5,q006,Which listed method has the strongest MS MARCO...,table,paper_02_page_004
6,q007,How does DSI turn document retrieval into a te...,text,paper_03_page_002
7,q008,Why can semantically structured document ident...,text,paper_03_page_006
8,q009,Which DSI variant performs best in the documen...,table,paper_03_page_008
9,q010,How does DSI performance change as model size ...,figure,paper_03_page_009



Questions by type:


,count
question_type,
text,18
table,17
figure,15


## Create `qrels.csv` and `gold_dict`

For the current single-gold-page setup:

```python
gold_dict = {
    "q001": "paper01_page_007",
    ...
}
```

and:

```text
qid,page_id,relevance
q001,paper01_page_007,1
```


In [ ]:
qrels = queries_clean[['qid', 'page_id']].copy()
qrels['relevance'] = 1

gold_dict = dict(
    zip(queries_clean['qid'], queries_clean['page_id'])
)

print("qrels:")
display(qrels.head())

print("\nExample gold_dict:")
print(list(gold_dict.items())[:5])


qrels:


,qid,page_id,relevance
0,q001,paper_01_page_002,1
1,q002,paper_01_page_004,1
2,q003,paper_01_page_005,1
3,q004,paper_01_page_008,1
4,q005,paper_02_page_002,1



Example gold_dict:
[('q001', 'paper_01_page_002'), ('q002', 'paper_01_page_004'), ('q003', 'paper_01_page_005'), ('q004', 'paper_01_page_008'), ('q005', 'paper_02_page_002')]


In [ ]:
# Consistency checks

page_ids = set(page_text_df['page_id'])
missing_gold = sorted(set(qrels['page_id']) - page_ids)

if missing_gold:
    raise ValueError(
        "Gold pages missing from page_text_df: " + str(missing_gold)
    )

print("PASS: every gold page exists in the Markdown corpus.")

if len(queries_clean) == 50:
    print("PASS: exactly 50 questions.")
else:
    print(f"WARNING: expected 50 questions, found {len(queries_clean)}.")


PASS: every gold page exists in the Markdown corpus.
PASS: exactly 50 questions.


## Save everything to Google Drive

The prepared data will be stored here:

```text
My Drive/
└── ColPali IR project/
    └── prepared_data/
        ├── pages.csv
        ├── queries_clean.csv
        ├── qrels.csv
        └── gold_dict.pkl
```




In [ ]:
import pickle

OUTPUT_DIR = PROJECT_ROOT / 'prepared_data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pages_csv = OUTPUT_DIR / 'pages.csv'
queries_csv = OUTPUT_DIR / 'queries_clean.csv'
qrels_csv = OUTPUT_DIR / 'qrels.csv'
gold_pkl = OUTPUT_DIR / 'gold_dict.pkl'

page_text_df[['page_id', 'page_text']].to_csv(pages_csv, index=False)
queries_clean.to_csv(queries_csv, index=False)
qrels.to_csv(qrels_csv, index=False)

with open(gold_pkl, 'wb') as f:
    pickle.dump(gold_dict, f)

print("Saved to Google Drive:")
for p in [pages_csv, queries_csv, qrels_csv, gold_pkl]:
    print(" ", p)


Saved to Google Drive:
  /content/drive/MyDrive/ColPali IR project/prepared_data/pages.csv
  /content/drive/MyDrive/ColPali IR project/prepared_data/queries_clean.csv
  /content/drive/MyDrive/ColPali IR project/prepared_data/qrels.csv
  /content/drive/MyDrive/ColPali IR project/prepared_data/gold_dict.pkl


In [ ]:
# Verify the saved files

for p in [pages_csv, queries_csv, qrels_csv, gold_pkl]:
    print(f"{p.name:20s} exists={p.exists()} size={p.stat().st_size if p.exists() else 'N/A'} bytes")

print("\nOutput folder:", OUTPUT_DIR)

print("\n========== SUMMARY ==========")
print("Pages:", len(page_text_df))
print("Questions:", len(queries_clean))
print("Question types:", queries_clean['question_type'].value_counts().to_dict())
print("Unique gold pages:", queries_clean['page_id'].nunique())


pages.csv            exists=True size=1006626 bytes
queries_clean.csv    exists=True size=5385 bytes
qrels.csv            exists=True size=1272 bytes
gold_dict.pkl        exists=True size=1366 bytes

Output folder: /content/drive/MyDrive/ColPali IR project/prepared_data

========== SUMMARY ==========
Pages: 310
Questions: 50
Question types: {'text': 18, 'table': 17, 'figure': 15}
Unique gold pages: 50


## Page mapping for the next ColQwen2/BM25 notebook

Keep the same filename stem for the page image and Markdown:

```text
paper01_page_007.md
paper01_page_007.png
```

Both map to:

```text
page_id = paper01_page_007
```

Then:

- ColQwen2 retrieves using the **page image**
- BM25 retrieves using the **page_text**
- both systems use the same `page_id`
- both are evaluated against the same `qrels.csv`


